# Memoria Persistente y Sistemas Multi-Agente

## Modulo 5 -- Agentes Avanzados | Semana 12

---

## Objetivos de Aprendizaje

| # | Objetivo |
|---|----------|
| 1 | Entender los **tipos de memoria** de un agente (corto plazo, largo plazo, scratchpad) |
| 2 | Implementar **conversation summarization** para controlar el crecimiento del historial |
| 3 | Construir **memoria persistente** entre conversaciones con SQLite |
| 4 | Disenar **sistemas multi-agente** con agentes especializados |
| 5 | Implementar los 3 patrones de comunicacion: **pipeline, delegacion y debate** |
| 6 | Manejar **shared state** entre agentes y evaluar **costo vs complejidad** |

---

> **Contexto**: Seguimos con **Klimbook**. En esta leccion el agente va a recordar cosas entre sesiones y vamos a dividir responsabilidades entre multiples agentes especializados.

---

## 12.1 El Problema de la Memoria

Hasta ahora, todos tus agentes son **stateless**. Cada conversacion empieza de cero. El agente no recuerda nada de conversaciones anteriores.

```
Conversacion 1:
  Usuario: "Mi proyecto de escalada favorito es La Esfinge 5.11a"
  Agente:  "Anotado! La Esfinge 5.11a es un gran proyecto."

Conversacion 2 (al dia siguiente):
  Usuario: "Como va mi proyecto?"
  Agente:  "No se a que proyecto te refieres."

  -> El agente no recuerda NADA de la conversacion anterior.
     Cada llamada a la API es independiente.
```

Esto pasa porque **la API de Claude es stateless**. Cada request es independiente. Claude no tiene memoria entre requests. La "memoria" que ves en claude.ai es el frontend que reenvía los mensajes anteriores en cada request.

### Tipos de memoria

| Tipo | Alcance | Ya lo tienes? | Ejemplo |
|------|---------|---------------|---------|
| **Corto plazo** | Dentro de UNA conversacion | Si (array de `messages`) | Cada step agrega assistant + tool_result |
| **Largo plazo** | Entre conversaciones | No | Preferencias del usuario, historial |
| **Scratchpad** | Durante una tarea compleja | No | Notas internas del agente para organizarse |

> **Problema del corto plazo**: el array de `messages` crece con cada step y consume tokens. Despues de 20 tool calls, el historial puede tener 50,000+ tokens. Cada nueva llamada envia TODO el historial.

---

## 12.2 Memoria de Corto Plazo: Conversation Summarization

El problema con la memoria de corto plazo (el array de `messages`) es que **crece linealmente**. Despues de 20 tool calls, puede tener 50,000+ tokens. Cada nueva llamada a la API envia TODO el historial -- costoso y eventualmente excede la ventana de contexto.

**Solucion**: resumir la conversacion periodicamente.

```
Sin resumen (50,000 tokens):
  [msg1] [msg2] [msg3] ... [msg48] [msg49] [msg50]
  -> Se envia TODO en cada request

Con resumen cada 8 iteraciones (~5,000 tokens):
  [resumen de msgs 1-40] [msg47] [msg48] [msg49] [msg50]
  -> Resumen comprime el historial viejo, conserva los recientes
```

In [ ]:
import json
import logging
from anthropic import AsyncAnthropic

client = AsyncAnthropic()
MODEL = "claude-sonnet-4-20250514"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


async def summarize_conversation(messages: list[dict]) -> str:
    """
    Resume la conversacion hasta ahora en un parrafo corto.

    Se usa cuando el historial de messages crece demasiado.
    En vez de enviar 50 mensajes, envias un resumen + los ultimos 5.

    El resumen captura:
    - Que pregunto el usuario originalmente
    - Que tools se llamaron y que resultados dieron
    - Que conclusiones se sacaron
    - Que queda pendiente
    """
    # Serializar los mensajes a texto para que Claude los resuma
    conversation_text = ""
    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        if isinstance(content, str):
            conversation_text += f"\n{role}: {content}"
        elif isinstance(content, list):
            # Tool results son listas de dicts
            for item in content:
                if isinstance(item, dict):
                    conversation_text += f"\n{role}: {json.dumps(item)[:300]}"
                else:
                    conversation_text += f"\n{role}: {str(item)[:300]}"
        else:
            conversation_text += f"\n{role}: {str(content)[:300]}"

    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        system="Summarize this conversation concisely. Include: "
        "the user's original question, tools used, key findings, "
        "and any pending tasks.",
        messages=[
            {
                "role": "user",
                "content": f"Summarize this conversation:\n{conversation_text}",
            }
        ],
        temperature=0.0,
        max_tokens=500,
    )

    return response.content[0].text


print("summarize_conversation() definida.")
print("Usa Haiku (barato y rapido) para comprimir el historial.")

### Agent Loop con Summarization

Integramos el resumen en el agent loop existente. Cada N iteraciones, comprime el historial:

In [ ]:
async def agent_loop_with_summarization(
    user_message: str,
    tools: list,
    max_iterations: int = 20,
    summarize_every: int = 8,
):
    """
    Agent loop que resume la conversacion cada N iteraciones
    para mantener el historial manejable.

    Cada summarize_every iteraciones:
    1. Llama a summarize_conversation() con el historial completo
    2. Reemplaza el historial con: resumen + ultimos 4 mensajes
    3. Esto reduce tokens de ~50K a ~5K
    """
    messages = [{"role": "user", "content": user_message}]

    for iteration in range(max_iterations):
        # Cada N iteraciones, resumir y comprimir el historial
        if iteration > 0 and iteration % summarize_every == 0:
            summary = await summarize_conversation(messages)

            # Reemplazar todo el historial con:
            # 1. El resumen como contexto
            # 2. Los ultimos 4 mensajes (contexto inmediato)
            recent = messages[-4:]
            messages = [
                {
                    "role": "user",
                    "content": (
                        f"Conversation summary so far:\n{summary}\n\n"
                        f"Continue from here. Original question: {user_message}"
                    ),
                },
            ] + recent

            logger.info(
                f"  [Memory] Conversacion resumida en iteracion {iteration}. "
                f"Historial comprimido a {len(messages)} mensajes."
            )

        # Agent loop normal
        response = await client.messages.create(
            model=MODEL,
            max_tokens=4096,
            tools=tools,
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            return response.content[0].text

        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            # process_tool_calls se define segun tus tools
            # tool_results = process_tool_calls(response)
            # messages.append({"role": "user", "content": tool_results})

    return "Max iterations reached."


print("agent_loop_with_summarization() definida.")
print("Resume cada 8 iteraciones para controlar tokens.")

---

## 12.3 Memoria de Largo Plazo: Persistencia entre Conversaciones

Para que el agente **recuerde cosas entre sesiones**, necesitas guardar informacion en algun lado y recuperarla al inicio de cada conversacion.

| Opcion | Complejidad | Ideal para | Requiere servidor? |
|--------|-------------|------------|-------------------|
| **Archivos JSON** | Baja | Prototyping rapido | No |
| **SQLite** | Media | Agente single-user | No (archivo en disco) |
| **PostgreSQL** | Alta | Produccion multi-user | Si |
| **Redis** | Media | Cache + sesiones rapidas | Si |

### Opcion 1: Archivos JSON (prototyping)

In [ ]:
from pathlib import Path
from datetime import datetime

MEMORY_FILE = Path("agent_memory.json")


def load_memory() -> dict:
    """Carga la memoria del agente desde disco."""
    if MEMORY_FILE.exists():
        return json.loads(MEMORY_FILE.read_text())
    return {
        "user_preferences": {},
        "conversation_summaries": [],
        "facts": [],
        "last_updated": None,
    }


def save_memory(memory: dict):
    """Guarda la memoria del agente en disco."""
    memory["last_updated"] = datetime.now().isoformat()
    MEMORY_FILE.write_text(json.dumps(memory, indent=2, ensure_ascii=False))


def remember(key: str, value: str):
    """Guarda un hecho en la memoria."""
    memory = load_memory()
    memory["facts"].append(
        {
            "key": key,
            "value": value,
            "timestamp": datetime.now().isoformat(),
        }
    )
    save_memory(memory)


def recall(key: str) -> str | None:
    """Busca un hecho en la memoria (mas reciente primero)."""
    memory = load_memory()
    for fact in reversed(memory["facts"]):
        if key.lower() in fact["key"].lower():
            return fact["value"]
    return None


# Demo
remember("proyecto_favorito", "La Esfinge 5.11a")
remember("nivel_escalada", "5.11a redpoint, 5.10d onsight")

print("Hechos guardados en memoria:")
memory = load_memory()
for fact in memory["facts"]:
    print(f"  {fact['key']}: {fact['value']}")

print(f"\nRecall 'proyecto': {recall('proyecto')}")
print(f"Recall 'nivel':    {recall('nivel')}")

### Opcion 2: SQLite (recomendado para desarrollo)

SQLite es un archivo en disco que actua como base de datos relacional. No necesitas instalar nada extra (viene con Python). Es perfecto para un agente **single-user**.

> **Ventaja sobre JSON**: consultas estructuradas, updates sin reescribir todo el archivo, manejo de concurrencia basico, y no corrompe datos si el proceso se interrumpe.

In [ ]:
import sqlite3


class AgentMemory:
    """
    Memoria persistente del agente usando SQLite.

    Tres tablas:
    - facts: hechos sobre el usuario (categoria/key/value)
    - conversation_history: resumenes de conversaciones pasadas
    - preferences: preferencias del usuario (key/value)
    """

    def __init__(self, db_path: str = "agent_memory.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row  # retorna dicts en vez de tuples
        self._create_tables()

    def _create_tables(self):
        """Crea las tablas si no existen."""
        self.conn.executescript(
            """
            -- Hechos que el agente recuerda sobre el usuario
            CREATE TABLE IF NOT EXISTS facts (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                category TEXT NOT NULL,
                key TEXT NOT NULL,
                value TEXT NOT NULL,
                confidence REAL DEFAULT 1.0,
                source TEXT DEFAULT 'user',
                created_at TEXT DEFAULT (datetime('now')),
                updated_at TEXT DEFAULT (datetime('now'))
            );

            -- Resumenes de conversaciones pasadas
            CREATE TABLE IF NOT EXISTS conversation_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                summary TEXT NOT NULL,
                topics TEXT,
                tools_used TEXT,
                created_at TEXT DEFAULT (datetime('now'))
            );

            -- Preferencias del usuario
            CREATE TABLE IF NOT EXISTS preferences (
                key TEXT PRIMARY KEY,
                value TEXT NOT NULL,
                updated_at TEXT DEFAULT (datetime('now'))
            );
        """
        )
        self.conn.commit()

    def remember_fact(self, category: str, key: str, value: str, source: str = "user"):
        """
        Guarda un hecho sobre el usuario.

        Si ya existe un hecho con la misma key en la misma categoria,
        lo actualiza. Esto evita duplicados.
        """
        existing = self.conn.execute(
            "SELECT id FROM facts WHERE category = ? AND key = ?", (category, key)
        ).fetchone()

        if existing:
            self.conn.execute(
                "UPDATE facts SET value = ?, updated_at = datetime('now') "
                "WHERE id = ?",
                (value, existing["id"]),
            )
        else:
            self.conn.execute(
                "INSERT INTO facts (category, key, value, source) "
                "VALUES (?, ?, ?, ?)",
                (category, key, value, source),
            )
        self.conn.commit()

    def recall_facts(self, category: str = None, limit: int = 20) -> list[dict]:
        """Recupera hechos. Si se especifica categoria, filtra por ella."""
        if category:
            rows = self.conn.execute(
                "SELECT * FROM facts WHERE category = ? "
                "ORDER BY updated_at DESC LIMIT ?",
                (category, limit),
            ).fetchall()
        else:
            rows = self.conn.execute(
                "SELECT * FROM facts ORDER BY updated_at DESC LIMIT ?",
                (limit,),
            ).fetchall()
        return [dict(row) for row in rows]

    def save_conversation(self, summary: str, topics: list[str], tools: list[str]):
        """Guarda el resumen de una conversacion."""
        self.conn.execute(
            "INSERT INTO conversation_history (summary, topics, tools_used) "
            "VALUES (?, ?, ?)",
            (summary, json.dumps(topics), json.dumps(tools)),
        )
        self.conn.commit()

    def get_recent_conversations(self, limit: int = 5) -> list[dict]:
        """Retorna las N conversaciones mas recientes."""
        rows = self.conn.execute(
            "SELECT * FROM conversation_history "
            "ORDER BY created_at DESC LIMIT ?",
            (limit,),
        ).fetchall()
        return [dict(row) for row in rows]

    def set_preference(self, key: str, value: str):
        """Guarda o actualiza una preferencia del usuario."""
        self.conn.execute(
            "INSERT OR REPLACE INTO preferences (key, value, updated_at) "
            "VALUES (?, ?, datetime('now'))",
            (key, value),
        )
        self.conn.commit()

    def get_preference(self, key: str) -> str | None:
        """Recupera una preferencia."""
        row = self.conn.execute(
            "SELECT value FROM preferences WHERE key = ?", (key,)
        ).fetchone()
        return row["value"] if row else None

    def get_context_for_prompt(self) -> str:
        """
        Genera un bloque de texto con toda la memoria relevante
        para inyectar en el system prompt del agente.

        Esto es lo que hace que el agente "recuerde" entre sesiones.
        """
        parts = []

        # Hechos sobre el usuario
        facts = self.recall_facts(limit=20)
        if facts:
            parts.append("Known facts about the user:")
            for f in facts:
                parts.append(f"- {f['category']}/{f['key']}: {f['value']}")

        # Preferencias
        prefs = self.conn.execute("SELECT * FROM preferences").fetchall()
        if prefs:
            parts.append("\nUser preferences:")
            for p in prefs:
                parts.append(f"- {p['key']}: {p['value']}")

        # Conversaciones recientes
        convos = self.get_recent_conversations(3)
        if convos:
            parts.append("\nRecent conversation summaries:")
            for c in convos:
                parts.append(f"- [{c['created_at']}] {c['summary'][:200]}")

        return "\n".join(parts) if parts else "No prior information about this user."


# ---- Demo ----

mem = AgentMemory(":memory:")  # en memoria para demo (normalmente usa archivo)

mem.remember_fact("climbing", "current_project", "La Esfinge 5.11a")
mem.remember_fact("climbing", "level", "5.11a redpoint, 5.10d onsight")
mem.remember_fact("personal", "name", "Carlos")
mem.set_preference("language", "es")
mem.set_preference("response_style", "concise")
mem.save_conversation(
    summary="User asked about bouldering gyms near Lima",
    topics=["climbing", "gyms", "Lima"],
    tools=["search_docs"],
)

print("=== Contexto para el prompt ===")
print(mem.get_context_for_prompt())

### Integrando la Memoria en el Agente

El patron es:
1. **Al inicio** de cada conversacion: cargar la memoria e inyectarla en el system prompt
2. **Durante** la conversacion: si el usuario comparte info nueva, el agente llama `remember_fact`
3. **Al final**: guardar un resumen de la conversacion

In [ ]:
memory = AgentMemory("klimbook_agent.db")


def build_system_prompt() -> str:
    """
    Construye el system prompt inyectando la memoria persistente.

    Cada vez que el agente inicia una conversacion, carga sus
    recuerdos y los incluye en el system prompt para que Claude
    los tenga como contexto.
    """
    context = memory.get_context_for_prompt()

    return f"""You are a helpful assistant for Klimbook.

You have memory of past interactions with this user:

<memory>
{context}
</memory>

Use this context naturally. Don't mention that you're reading from memory.
If you learn new facts about the user during the conversation, call the
remember_fact tool to save them for next time.

Available tools for memory management:
- remember_fact: save a new fact about the user
- recall_facts: retrieve stored facts"""


# Tools de memoria que el agente puede usar DURANTE la conversacion
MEMORY_TOOLS = [
    {
        "name": "remember_fact",
        "description": (
            "Save a fact about the user for future conversations. "
            "Use this when the user shares personal information, preferences, "
            "or important context that would be useful to remember later. "
            "Categories: 'personal', 'climbing', 'technical', 'preferences'. "
            "Example: remember_fact(category='climbing', key='current_project', "
            "value='La Esfinge 5.11a')"
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "enum": ["personal", "climbing", "technical", "preferences"],
                },
                "key": {"type": "string"},
                "value": {"type": "string"},
            },
            "required": ["category", "key", "value"],
        },
    },
    {
        "name": "recall_facts",
        "description": (
            "Retrieve stored facts about the user. "
            "Use when you need to reference something from past conversations."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "description": "Filter by category (optional)",
                },
            },
        },
    },
]


print("build_system_prompt() definida.")
print("MEMORY_TOOLS definidos: remember_fact, recall_facts")
print(f"\nEjemplo de system prompt generado:\n")
print(build_system_prompt()[:400] + "...")

---

## 12.4 Sistemas Multi-Agente

Hasta ahora, un solo agente hace todo. Pero para tareas complejas, es mas efectivo tener **varios agentes especializados** que colaboran.

### Por que multi-agente?

| Aspecto | Agente unico | Multi-agente |
|---------|-------------|--------------|
| **System prompt** | Enorme, con 50 tools | Cada agente tiene 3-5 tools enfocadas |
| **Confusion** | Claude se confunde sobre que tool usar | Cada agente sabe exactamente su dominio |
| **Debugging** | Todo mezclado, dificil de aislar | Cada agente es independiente y testeable |
| **Modelos** | Un solo modelo para todo | Haiku para routing, Sonnet para razonamiento |
| **Desarrollo** | Cambiar algo puede romper todo | Desarrollar y probar cada agente por separado |

### Arquitectura: Klimbook Support System

```
                    +-------------------+
                    |   Router Agent    |  <- Haiku (clasifica el request)
                    +--------+----------+
                             |
              +--------------+--------------+
              |              |              |
    +---------v---+  +-------v------+  +----v---------+
    | Docs Agent  |  | Data Agent   |  | Action Agent |
    | (RAG)       |  | (SQL)        |  | (GitHub)     |
    +-------------+  +--------------+  +--------------+

    Docs Agent:   search_docs         (Modelo: Sonnet)
    Data Agent:   list_tables,        (Modelo: Sonnet)
                  get_schema,
                  query_db
    Action Agent: create_issue,       (Modelo: Sonnet)
                  list_repos
```

### Router Agent

El router **no resuelve la tarea**. Solo decide **quien** la resuelve. Es ligero (Haiku) y rapido porque solo clasifica.

In [ ]:
from anthropic import Anthropic

sync_client = Anthropic()


async def route_request(message: str) -> str:
    """
    Clasifica el mensaje y lo envia al agente especializado correcto.

    El router NO resuelve la tarea. Solo decide QUIEN la resuelve.
    Usa Haiku porque es rapido y barato -- solo necesita clasificar.
    """
    # Paso 1: Clasificar con Haiku
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        system="""Classify this user message into ONE category:
- docs: questions about how Klimbook works, architecture, API, features
- data: questions about statistics, numbers, user data, reports
- action: requests to create issues, manage releases, modify something
- general: greetings, small talk, questions unrelated to Klimbook

Respond with ONLY the category name.""",
        messages=[{"role": "user", "content": message}],
        temperature=0.0,
        max_tokens=20,
    )

    category = response.content[0].text.strip().lower()
    logger.info(f"[Router] Classified as: {category}")

    # Paso 2: Despachar al agente correcto
    if category == "docs":
        return await docs_agent(message)
    elif category == "data":
        return await data_agent(message)
    elif category == "action":
        return await action_agent(message)
    else:
        return await general_agent(message)


print("route_request() definida.")
print("Clasifica con Haiku y despacha al agente especializado.")

### Agentes Especializados

Cada agente tiene su propio system prompt enfocado, sus propias tools, y potencialmente su propio modelo.

In [ ]:
# Placeholder para run_agent (el agent loop generico).
# En un proyecto real, seria el agent loop completo de la leccion 6.

async def run_agent(
    message: str,
    tools: list,
    system: str,
    model: str = "claude-sonnet-4-20250514",
    tool_functions: dict = None,
) -> str:
    """Agent loop generico reutilizable por todos los agentes."""
    messages = [{"role": "user", "content": message}]

    for _ in range(10):
        response = await client.messages.create(
            model=model,
            max_tokens=4096,
            tools=tools or [],
            system=system,
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            return response.content[0].text

        if response.stop_reason == "tool_use" and tool_functions:
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    func = tool_functions[block.name]
                    result = await func(**block.input) if callable(func) else func
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result, ensure_ascii=False),
                        }
                    )
            messages.append({"role": "user", "content": tool_results})

    return "Max iterations reached."


# Placeholder tools (en un proyecto real, importarias de otros modulos)
RAG_TOOLS = []     # search_docs (de leccion 11)
DB_TOOLS = []      # list_tables, get_schema, query_db (de leccion 7)
GITHUB_TOOLS = []  # create_issue, list_repos


async def docs_agent(message: str) -> str:
    """Agente especializado en documentacion. Usa RAG."""
    return await run_agent(
        message=message,
        tools=RAG_TOOLS,
        system="""You are a documentation expert for Klimbook.
Use search_docs to find relevant documentation before answering.
Only answer based on what the documentation says.""",
        model="claude-sonnet-4-20250514",
    )


async def data_agent(message: str) -> str:
    """Agente especializado en datos. Usa SQL."""
    return await run_agent(
        message=message,
        tools=DB_TOOLS,
        system="""You are a data analyst for Klimbook.
Use the database tools to answer questions about user data,
climbing statistics, and generate reports.
Always check the schema before writing queries.""",
        model="claude-sonnet-4-20250514",
    )


async def action_agent(message: str) -> str:
    """Agente especializado en acciones. Interactua con GitHub."""
    return await run_agent(
        message=message,
        tools=GITHUB_TOOLS,
        system="""You are a project manager for Klimbook.
Help with GitHub operations like creating issues, checking commits,
and managing releases. Always confirm before creating or modifying.""",
        model="claude-sonnet-4-20250514",
    )


async def general_agent(message: str) -> str:
    """Agente para conversacion general (sin tools)."""
    return await run_agent(
        message=message,
        tools=[],
        system="You are a friendly assistant for Klimbook users.",
        model="claude-haiku-4-5-20251001",  # Haiku basta para small talk
    )


print("Agentes especializados definidos:")
print("  - docs_agent()    -> RAG, busca en documentacion")
print("  - data_agent()    -> SQL, consulta base de datos")
print("  - action_agent()  -> GitHub, crea issues/releases")
print("  - general_agent() -> Small talk, sin tools")

---

## 12.5 Patrones de Comunicacion entre Agentes

Los agentes pueden comunicarse de varias formas. Los tres patrones mas comunes:

| Patron | Flujo | Ideal para | Ejemplo |
|--------|-------|------------|---------|
| **Pipeline** | A -> B -> C (secuencial) | Tareas con pasos definidos | Release notes: clasificar -> escribir -> revisar |
| **Delegacion** | Orquestador asigna subtareas | Tareas complejas multi-dominio | Support: router -> docs/data/action agents |
| **Debate** | Generar + Revisar | Mejorar calidad | Codigo: generar -> code review -> version final |

### Patron 1: Pipeline (secuencial)

Un agente pasa su output al siguiente. Como una linea de ensamble.

```
Agente A -> output -> Agente B -> output -> Agente C -> resultado final

Ejemplo: Release Notes Pipeline
  Git Agent (lee commits)
    -> Classifier (clasifica por tipo)
      -> Writer (genera notas)
        -> Reviewer (valida y mejora)
```

In [ ]:
async def pipeline_agents(commits_text: str) -> str:
    """
    Pipeline de agentes secuencial.

    Cada agente recibe el output del anterior y lo transforma.
    El resultado final pasa por 3 etapas de procesamiento.
    """
    # Agente 1: Clasificar commits por tipo (Haiku -- tarea simple)
    classified = await run_agent(
        message=f"Classify these commits by type (feat/fix/docs/chore):\n{commits_text}",
        tools=[],
        system="You classify git commits by type. Output a structured list.",
        model="claude-haiku-4-5-20251001",
    )

    # Agente 2: Generar notas (Sonnet -- necesita buena redaccion)
    notes = await run_agent(
        message=f"Generate professional release notes from:\n{classified}",
        tools=[],
        system="You write clear, professional release notes for developers.",
        model="claude-sonnet-4-20250514",
    )

    # Agente 3: Revisar (Sonnet -- necesita juicio critico)
    reviewed = await run_agent(
        message=(
            f"Review these release notes and suggest improvements:\n{notes}\n\n"
            f"Check for: clarity, accuracy, completeness, and tone."
        ),
        tools=[],
        system="You are a technical editor. Be thorough but constructive.",
        model="claude-sonnet-4-20250514",
    )

    return reviewed


print("pipeline_agents() definida.")
print("Flujo: commits -> classify (Haiku) -> write (Sonnet) -> review (Sonnet)")

### Patron 2: Delegacion (jerarquico)

Un agente "jefe" (orquestador) delega subtareas a agentes "trabajadores". El orquestador decide que hacer, asigna tareas, recibe resultados, y sintetiza la respuesta final.

```
       Orchestrator
       /    |     \
    Agent   Agent   Agent
      A       B       C

El orchestrator ve a cada agente como una tool mas.
Cuando llama "ask_data_analyst", tu codigo ejecuta
el data_agent completo (con su propio agent loop).
```

In [ ]:
async def orchestrator_agent(user_request: str) -> str:
    """
    Agente orquestador que delega subtareas a agentes especializados.

    Las tools del orquestador son los otros agentes.
    Desde su perspectiva, cada agente es simplemente una tool.
    """
    # Tools que representan a los sub-agentes
    ORCHESTRATOR_TOOLS = [
        {
            "name": "ask_docs_expert",
            "description": "Ask the documentation expert a question about Klimbook.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                },
                "required": ["question"],
            },
        },
        {
            "name": "ask_data_analyst",
            "description": "Ask the data analyst to query the database.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                },
                "required": ["question"],
            },
        },
        {
            "name": "ask_github_manager",
            "description": "Ask the GitHub manager to perform an action.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "request": {"type": "string"},
                },
                "required": ["request"],
            },
        },
    ]

    # Funciones que ejecutan los sub-agentes completos
    async def ask_docs_expert(question: str) -> dict:
        answer = await docs_agent(question)
        return {"answer": answer}

    async def ask_data_analyst(question: str) -> dict:
        answer = await data_agent(question)
        return {"answer": answer}

    async def ask_github_manager(request: str) -> dict:
        answer = await action_agent(request)
        return {"answer": answer}

    TOOL_FUNCTIONS = {
        "ask_docs_expert": ask_docs_expert,
        "ask_data_analyst": ask_data_analyst,
        "ask_github_manager": ask_github_manager,
    }

    # El orquestador es un agent loop normal.
    # Cuando llama "ask_data_analyst", se ejecuta el data_agent
    # completo (con su propio agent loop, tools, etc.) y retorna
    # el resultado como tool_result.
    return await run_agent(
        message=user_request,
        tools=ORCHESTRATOR_TOOLS,
        tool_functions=TOOL_FUNCTIONS,
        system="""You are the orchestrator for Klimbook's support system.
Break down complex requests into subtasks and delegate to specialists:
- ask_docs_expert: for documentation questions
- ask_data_analyst: for data/statistics questions
- ask_github_manager: for GitHub operations

Synthesize the results into a coherent response for the user.""",
        model="claude-sonnet-4-20250514",
    )


print("orchestrator_agent() definida.")
print("Delega a sub-agentes: docs, data, github.")

### Patron 3: Debate / Verificacion

Dos agentes trabajan en la misma tarea: uno **genera**, otro **revisa**. Esto mejora la calidad porque el revisor detecta errores que el generador no vio.

```
       Tarea
      /     \
  Agent A   Agent B
  (genera)  (revisa)
      \     /
     Comparar
        |
    Resultado final
```

> **Cuando usarlo**: generacion de codigo, respuestas criticas (medicas, legales), documentacion tecnica donde la precision importa mucho.

In [ ]:
async def generate_and_review(task: str) -> str:
    """
    Un agente genera, otro revisa.

    El revisor verifica:
    1. Accuracy de la informacion
    2. Completeness (puntos faltantes?)
    3. Clarity y organizacion
    4. Tone apropiado

    Si el draft es bueno, lo retorna tal cual.
    Si necesita mejoras, retorna la version mejorada.
    """
    # Agente 1: Genera la respuesta
    draft = await run_agent(
        message=task,
        tools=[],
        system="Generate a thorough response to the task.",
        model="claude-sonnet-4-20250514",
    )

    # Agente 2: Revisa y mejora
    review = await run_agent(
        message=f"""Review this draft and improve it:

<draft>
{draft}
</draft>

Original task: {task}

Check for:
1. Accuracy of information
2. Completeness (any missing points?)
3. Clarity and organization
4. Tone appropriateness

If the draft is good, return it as-is.
If it needs improvements, return the improved version.""",
        tools=[],
        system="You are a critical reviewer. Be thorough but constructive.",
        model="claude-sonnet-4-20250514",
    )

    return review


print("generate_and_review() definida.")
print("Patron: generar con Sonnet -> revisar con Sonnet.")

---

## 12.6 Shared State entre Agentes

Cuando multiples agentes trabajan en la misma tarea, necesitan **compartir estado**. Sin estado compartido, cada agente trabaja en un silo y puede contradecir a los otros.

El `SharedState` es el **"pizarron"** donde los agentes anotan sus hallazgos para que los demas los vean.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class SharedState:
    """
    Estado compartido entre agentes.

    Cada agente puede leer y escribir en este objeto.
    Es el "pizarron" donde los agentes anotan sus hallazgos.
    """

    # Contexto de la tarea original
    original_request: str = ""

    # Hallazgos de cada agente (agente_name -> finding)
    findings: dict[str, str] = field(default_factory=dict)

    # Datos recopilados (resultados de queries, docs, etc.)
    data: dict = field(default_factory=dict)

    # Log de acciones realizadas (para debugging)
    action_log: list[dict] = field(default_factory=list)

    # Estado de la tarea
    status: str = "in_progress"  # in_progress, completed, failed

    def add_finding(self, agent_name: str, finding: str):
        """Un agente agrega un hallazgo."""
        self.findings[agent_name] = finding
        self.action_log.append(
            {
                "agent": agent_name,
                "action": "add_finding",
                "timestamp": datetime.now().isoformat(),
                "content": finding[:200],
            }
        )

    def add_data(self, key: str, value):
        """Guarda datos que otros agentes pueden usar."""
        self.data[key] = value

    def to_context(self) -> str:
        """
        Genera un resumen del estado compartido para inyectar
        en el prompt de cualquier agente.
        """
        parts = [f"Original request: {self.original_request}"]

        if self.findings:
            parts.append("\nFindings from other agents:")
            for agent, finding in self.findings.items():
                parts.append(f"  [{agent}]: {finding[:300]}")

        if self.data:
            parts.append(f"\nShared data keys: {list(self.data.keys())}")

        return "\n".join(parts)


# ---- Uso con multiples agentes ----

async def multi_agent_task(request: str) -> str:
    """Ejecuta una tarea con multiples agentes que comparten estado."""

    state = SharedState(original_request=request)

    # Paso 1: Docs agent investiga
    docs_result = await docs_agent(
        f"Based on Klimbook's docs, research: {request}"
    )
    state.add_finding("docs_agent", docs_result)

    # Paso 2: Data agent consulta datos.
    # PUEDE VER lo que el docs_agent encontro.
    data_result = await data_agent(
        f"Context from docs: {state.findings.get('docs_agent', '')[:500]}\n\n"
        f"Now query the database for: {request}"
    )
    state.add_finding("data_agent", data_result)

    # Paso 3: Synthesizer combina todo
    synthesis = await run_agent(
        message=(
            f"Synthesize these findings into a final answer:\n\n"
            f"{state.to_context()}"
        ),
        tools=[],
        system="Combine findings from multiple sources into a clear, "
        "coherent response. Resolve any contradictions.",
        model="claude-sonnet-4-20250514",
    )

    state.status = "completed"
    return synthesis


# Demo del SharedState
demo_state = SharedState(original_request="How many users climbed 5.12+ last month?")
demo_state.add_finding("docs_agent", "Routes are graded using YDS system (5.0-5.15d)")
demo_state.add_finding("data_agent", "427 users logged routes of 5.12a or harder")
demo_state.add_data("user_count", 427)

print("=== Shared State Context ===")
print(demo_state.to_context())
print(f"\nAction log: {len(demo_state.action_log)} entries")
for entry in demo_state.action_log:
    print(f"  [{entry['agent']}] {entry['content'][:60]}")

---

## 12.7 Consideraciones de Costo y Latencia

Los sistemas multi-agente son **mas caros y lentos** porque hacen multiples llamadas a la API. Evaluemos el impacto:

### Comparacion de costos

| Arquitectura | Requests API | Costo aprox. | Latencia aprox. |
|-------------|-------------|-------------|-----------------|
| **Agente unico** | 1 a Sonnet | ~$0.01 | ~2s |
| **Pipeline 3 etapas** | 3 a Sonnet | ~$0.03 | ~6s |
| **Router + 1 sub-agente** | 1 Haiku + 1 Sonnet + tool calls | ~$0.05 | ~8s |
| **Orquestador + 2 sub-agentes** (3 tool calls c/u) | 1 + 2*(1 + 3*2) = 15 requests | ~$0.15 | ~30s |

### Cuando usar cada arquitectura

| Usa **agente unico** cuando... | Usa **multi-agente** cuando... |
|-------------------------------|-------------------------------|
| Tarea simple (1-3 tool calls) | Tarea compleja, multiples dominios |
| System prompt cabe en ~500 palabras | Tendrias 20+ tools en un solo agente |
| Menos de 10 tools | Agentes necesitan modelos diferentes |
| Latencia importa (usuario esperando) | Calidad importa mas que velocidad |
| Prototyping rapido | Quieres modularidad para desarrollo/testing |

> **Regla practica**: empieza siempre con un agente unico. Solo migra a multi-agente cuando el agente unico se vuelve dificil de mantener o la calidad baja por tener demasiadas tools.

---

## Resumen Semana 12

### Arquitectura General

```
                    +-------------------+
                    |   Router (Haiku)  |
                    +--------+----------+
                             |
              +--------------+--------------+
              |              |              |
    +---------v---+  +-------v------+  +----v---------+
    | Docs Agent  |  | Data Agent   |  | Action Agent |
    | (RAG)       |  | (SQL)        |  | (GitHub)     |
    +------+------+  +------+-------+  +------+-------+
           |                |                  |
           +--------+-------+------------------+
                    |
             Shared State
             (pizarron)
                    |
           +--------v--------+
           | Memoria (SQLite) |  <- Persiste entre conversaciones
           +-----------------+
```

### Conceptos Clave

| Concepto | Que es | Implementacion |
|----------|--------|----------------|
| **Memoria de corto plazo** | El array de `messages` que crece cada step | `summarize_conversation()` cada N iteraciones |
| **Memoria de largo plazo** | Hechos persistentes entre sesiones | `AgentMemory` con SQLite (facts, prefs, history) |
| **Tools de memoria** | El agente decide que guardar | `remember_fact`, `recall_facts` como tools |
| **Router** | Clasifica y despacha al agente correcto | Haiku (rapido + barato) |
| **Pipeline** | Agentes secuenciales (A -> B -> C) | Cada output es input del siguiente |
| **Delegacion** | Orquestador + trabajadores | Sub-agentes como tools del orquestador |
| **Debate** | Generar + revisar | Mejora calidad con doble pasada |
| **Shared State** | Pizarron compartido entre agentes | `SharedState` dataclass con findings/data/log |

### Checklist de Implementacion

```
Memoria:
[ ] Implementar summarize_conversation() para controlar tokens
[ ] Crear AgentMemory con SQLite (facts, preferences, history)
[ ] Agregar remember_fact y recall_facts como tools del agente
[ ] Inyectar memoria en el system prompt con get_context_for_prompt()
[ ] Guardar resumen al final de cada conversacion

Multi-agente:
[ ] Identificar si la tarea justifica multiples agentes
[ ] Implementar router con Haiku para clasificacion
[ ] Crear agentes especializados con 3-5 tools cada uno
[ ] Elegir patron de comunicacion (pipeline/delegacion/debate)
[ ] Implementar SharedState si los agentes necesitan colaborar
[ ] Medir costo y latencia vs agente unico
```

### Modelo Mental

```
"Cuando decidir que tipo de memoria usar:
 - Mismo request:     messages array (ya lo tienes)
 - Misma conversacion pero se hizo larga: summarize + recientes
 - Entre conversaciones: SQLite + inyectar en system prompt

 Cuando decidir si usar multi-agente:
 - <10 tools + task simple:  agente unico
 - 10-20 tools + dominios distintos:  router + agentes
 - Task compleja + calidad critica:  pipeline o debate"
```